In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment


def maxweight_schedule(q):
    """
    MaxWeight matching via Hungarian algorithm.
    We maximize sum_i q[i, p[i]] by minimizing -q.
    """
    row_ind, col_ind = linear_sum_assignment(-q)
    S = np.zeros_like(q, dtype=int)
    S[row_ind, col_ind] = 1
    return S


def sample_arrivals(n, epsilon, rng, mode="independent"):
    """
    Generate one arrival matrix A_t.

    mode = "synchronous":
        with probability (1-epsilon)/n, all entries are 1 simultaneously;
        otherwise all entries are 0.

    mode = "independent":
        each entry is iid Bernoulli((1-epsilon)/n).
    """
    p = (1.0 - epsilon) / n

    if mode == "synchronous":
        z = rng.binomial(1, p)
        return np.full((n, n), z, dtype=int)

    elif mode == "independent":
        return rng.binomial(1, p, size=(n, n))

    else:
        raise ValueError(f"Unknown mode: {mode}")


def simulate_iqs_maxweight_arrival_mode(n=4, epsilon=0.1, T=10000, seed=None, mode="independent"):
    rng = np.random.default_rng(seed)
    q = np.zeros((n, n), dtype=int)
    total_q = np.zeros(T, dtype=float)

    for t in range(T):
        S = maxweight_schedule(q)
        A = sample_arrivals(n=n, epsilon=epsilon, rng=rng, mode=mode)
        served = np.minimum(q, S)
        q = q - served + A
        total_q[t] = q.sum()

    running_max_q = np.maximum.accumulate(total_q)
    return total_q, running_max_q


def estimate_expected_curves_mode(n=4, epsilon=0.1, T=10000, num_runs=50, base_seed=0, mode="independent"):
    all_q = np.zeros((num_runs, T), dtype=float)
    all_running_max = np.zeros((num_runs, T), dtype=float)

    for k in range(num_runs):
        total_q, running_max_q = simulate_iqs_maxweight_arrival_mode(
            n=n,
            epsilon=epsilon,
            T=T,
            seed=base_seed + k,
            mode=mode,
        )
        all_q[k] = total_q
        all_running_max[k] = running_max_q

    mean_q = all_q.mean(axis=0)
    std_q = all_q.std(axis=0, ddof=1) if num_runs > 1 else np.zeros(T)

    mean_running_max = all_running_max.mean(axis=0)
    std_running_max = all_running_max.std(axis=0, ddof=1) if num_runs > 1 else np.zeros(T)

    return mean_q, std_q, mean_running_max, std_running_max


def apply_scientific_xaxis(ax, T):
    """
    Show x-axis like 0.0, 0.2, ..., 1.0 with a 1e5 / 1e6 style multiplier.
    """
    ticks = np.linspace(0, T, 6)
    ax.set_xticks(ticks)
    sci_exp = int(np.floor(np.log10(T))) if T > 0 else 0
    ax.ticklabel_format(style="sci", axis="x", scilimits=(sci_exp, sci_exp))


def plot_with_std(

    t_grid,

    mean1,

    std1,

    mean2,

    std2,

    ylabel,

    title,

    label1,

    label2,

    pdf_name,

    std_scale=1.0,

):

    fig, ax = plt.subplots(figsize=(8, 4))

    # 自定义颜色

    color1 = "#6A3D9A"   # 深紫

    color2 = "#1B9E77"   # 青绿

    lower1 = np.maximum(mean1 - std_scale * std1, 0.0)

    upper1 = mean1 + std_scale * std1

    lower2 = np.maximum(mean2 - std_scale * std2, 0.0)

    upper2 = mean2 + std_scale * std2

    ax.fill_between(t_grid, lower1, upper1, color=color1, alpha=0.18)

    ax.fill_between(t_grid, lower2, upper2, color=color2, alpha=0.18)

    ax.plot(t_grid, mean1, color=color1, linewidth=1.8, label=label1)

    ax.plot(t_grid, mean2, color=color2, linewidth=1.8, label=label2)

    ax.set_xlabel("T")

    ax.set_ylabel(ylabel)

    ax.set_title(title)

    ax.legend()

    ax.grid(alpha=0.3)

    apply_scientific_xaxis(ax, t_grid[-1])

    fig.tight_layout()

    fig.savefig(pdf_name, format="pdf", bbox_inches="tight")

    print(f"Saved figure to {pdf_name}")

    plt.show()


def run_comparison(n=4, T=100000, num_runs=50, epsilon=0.01, std_scale=1.0):
    print(f"n = {n}, epsilon = {epsilon}")
    print(f"Entrywise marginal arrival probability = (1-epsilon)/n = {(1.0 - epsilon)/n:.6f}")
    print("Comparing:")
    print("  1) synchronous arrivals")
    print("  2) independent Bernoulli arrivals")

    mean_q_sync, std_q_sync, mean_max_sync, std_max_sync = estimate_expected_curves_mode(
        n=n,
        epsilon=epsilon,
        T=T,
        num_runs=num_runs,
        base_seed=10000,
        mode="synchronous",
    )

    mean_q_ind, std_q_ind, mean_max_ind, std_max_ind = estimate_expected_curves_mode(
        n=n,
        epsilon=epsilon,
        T=T,
        num_runs=num_runs,
        base_seed=20000,
        mode="independent",
    )

    t_grid = np.arange(1, T + 1)
    log_t = np.log(t_grid + 1)

    # 1) E[Q_t]
    plot_with_std(
        t_grid=t_grid,
        mean1=mean_q_sync,
        std1=std_q_sync,
        mean2=mean_q_ind,
        std2=std_q_ind,
        ylabel=r"$\mathbb{E}[Q_t]$",
        title=fr"$\epsilon={epsilon}$",
        label1="synchronous arrivals",
        label2="independent Bernoulli arrivals",
        pdf_name=f"mean_queue_n_{n}_eps_{epsilon}.pdf",
        std_scale=std_scale,
    )

    # 2) E[max Q]
    plot_with_std(
        t_grid=t_grid,
        mean1=mean_max_sync,
        std1=std_max_sync,
        mean2=mean_max_ind,
        std2=std_max_ind,
        ylabel=r"$\mathbb{E}\!\left[\max_{0\leq t\leq T}\|Q_t\|_1 \right]$",
        title=fr"$\epsilon={epsilon}$",
        label1="Synchronous",
        label2="Independent",
        pdf_name=f"mean_running_max_{n}_eps_{epsilon}.pdf",
        std_scale=std_scale,
    )

    # 3) E[max Q]/log(t+1)
    plot_with_std(
        t_grid=t_grid,
        mean1=mean_max_sync / log_t,
        std1=std_max_sync / log_t,
        mean2=mean_max_ind / log_t,
        std2=std_max_ind / log_t,
        ylabel=r"$\mathbb{E}[\max_{s\leq t} Q_s]/\log(t+1)$",
        title=fr"$\epsilon={epsilon}$",
        label1="synchronous arrivals",
        label2="independent Bernoulli arrivals",
        pdf_name=f"mean_running_max_over_log_n_{n}_eps_{epsilon}.pdf",
        std_scale=std_scale,
    )


# example
run_comparison(
    n=4,
    T=100000,
    num_runs=200,
    epsilon=0.01,
    std_scale=0.25,
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment


def maxweight_schedule(q):
    """
    MaxWeight matching via Hungarian algorithm.
    We maximize sum_i q[i, p[i]] by minimizing -q.
    """
    row_ind, col_ind = linear_sum_assignment(-q)
    S = np.zeros_like(q, dtype=int)
    S[row_ind, col_ind] = 1
    return S


def check_capacity_profile(nu, tol=1e-12):
    return (
        np.all(nu >= -tol)
        and np.all(nu.sum(axis=1) <= 1 + tol)
        and np.all(nu.sum(axis=0) <= 1 + tol)
    )


def make_crp_profile(n):
    """
    CRP profile:
      - first row all 1/n
      - diagonal all 1/n
      - others 1/n^2
    """
    nu = np.full((n, n), 1.0 / (n * n), dtype=float)
    nu[0, :] = 1.0 / n
    for i in range(n):
        nu[i, i] = 1.0 / n
    return nu


def make_noncrp_profile(n):
    """
    non-CRP profile:
      - first row all 1/n
      - first column all 1/n
      - others 1/n^2
    """
    nu = np.full((n, n), 1.0 / (n * n), dtype=float)
    nu[0, :] = 1.0 / n
    nu[:, 0] = 1.0 / n
    return nu


def simulate_iqs_maxweight_profile(nu, epsilon=0.1, T=10000, seed=None):
    n = nu.shape[0]
    lam = (1.0 - epsilon) * nu

    rng = np.random.default_rng(seed)
    q = np.zeros((n, n), dtype=int)
    total_q = np.zeros(T, dtype=float)

    for t in range(T):
        S = maxweight_schedule(q)
        A = rng.binomial(1, lam)
        served = np.minimum(q, S)
        q = q - served + A
        total_q[t] = q.sum()

    running_max_q = np.maximum.accumulate(total_q)
    return total_q, running_max_q


def estimate_expected_curves_profile(nu, epsilon=0.1, T=10000, num_runs=50, base_seed=0):
    all_q = np.zeros((num_runs, T), dtype=float)
    all_running_max = np.zeros((num_runs, T), dtype=float)

    for k in range(num_runs):
        total_q, running_max_q = simulate_iqs_maxweight_profile(
            nu=nu,
            epsilon=epsilon,
            T=T,
            seed=base_seed + k,
        )
        all_q[k] = total_q
        all_running_max[k] = running_max_q

    mean_q = all_q.mean(axis=0)
    std_q = all_q.std(axis=0, ddof=1) if num_runs > 1 else np.zeros(T)

    mean_running_max = all_running_max.mean(axis=0)
    std_running_max = all_running_max.std(axis=0, ddof=1) if num_runs > 1 else np.zeros(T)

    return mean_q, std_q, mean_running_max, std_running_max


def apply_scientific_xaxis(ax, T):
    """
    Show x-axis like 0.0, 0.2, ..., 1.0 with a 1e5 / 1e6 style multiplier.
    """
    ticks = np.linspace(0, T, 6)
    ax.set_xticks(ticks)
    sci_exp = int(np.floor(np.log10(T))) if T > 0 else 0
    ax.ticklabel_format(style="sci", axis="x", scilimits=(sci_exp, sci_exp))


def plot_with_std(
    t_grid,
    mean1,
    std1,
    mean2,
    std2,
    ylabel,
    title,
    label1,
    label2,
    pdf_name,
    std_scale=1.0,
):
    fig, ax = plt.subplots(figsize=(8, 4))

    lower1 = np.maximum(mean1 - std_scale * std1, 0.0)
    upper1 = mean1 + std_scale * std1
    lower2 = np.maximum(mean2 - std_scale * std2, 0.0)
    upper2 = mean2 + std_scale * std2

    ax.fill_between(t_grid, lower1, upper1, alpha=0.18)
    ax.fill_between(t_grid, lower2, upper2, alpha=0.18)

    ax.plot(t_grid, mean1, linewidth=1.8, label=label1)
    ax.plot(t_grid, mean2, linewidth=1.8, label=label2)

    ax.set_xlabel("T")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)

    apply_scientific_xaxis(ax, t_grid[-1])
    fig.tight_layout()
    fig.savefig(pdf_name, format="pdf", bbox_inches="tight")
    print(f"Saved figure to {pdf_name}")
    plt.show()


def run_comparison(n=3, T=100000, num_runs=50, epsilon=0.01, std_scale=1.0):
    nu_crp = make_crp_profile(n)
    nu_noncrp = make_noncrp_profile(n)

    if not check_capacity_profile(nu_crp):
        raise ValueError(
            f"CRP profile invalid for n={n}. "
            f"row sums={nu_crp.sum(axis=1)}, col sums={nu_crp.sum(axis=0)}"
        )
    if not check_capacity_profile(nu_noncrp):
        raise ValueError(
            f"non-CRP profile invalid for n={n}. "
            f"row sums={nu_noncrp.sum(axis=1)}, col sums={nu_noncrp.sum(axis=0)}"
        )

    print(f"n = {n}, epsilon = {epsilon}")
    print("CRP row sums:", nu_crp.sum(axis=1))
    print("CRP col sums:", nu_crp.sum(axis=0))
    print("non-CRP row sums:", nu_noncrp.sum(axis=1))
    print("non-CRP col sums:", nu_noncrp.sum(axis=0))

    mean_q_crp, std_q_crp, mean_max_crp, std_max_crp = estimate_expected_curves_profile(
        nu=nu_crp,
        epsilon=epsilon,
        T=T,
        num_runs=num_runs,
        base_seed=10000,
    )

    mean_q_noncrp, std_q_noncrp, mean_max_noncrp, std_max_noncrp = estimate_expected_curves_profile(
        nu=nu_noncrp,
        epsilon=epsilon,
        T=T,
        num_runs=num_runs,
        base_seed=20000,
    )

    t_grid = np.arange(1, T + 1)
    log_t = np.log(t_grid + 1)

    # 1) E[Q_t]
    plot_with_std(
        t_grid=t_grid,
        mean1=mean_q_crp,
        std1=std_q_crp,
        mean2=mean_q_noncrp,
        std2=std_q_noncrp,
        ylabel=r"$\mathbb{E}[Q_t]$",
        title=f"Mean queue length, n={n}",
        label1=fr"CRP, $\epsilon={epsilon}$",
        label2=fr"non-CRP, $\epsilon={epsilon}$",
        pdf_name=f"mean_queue_n_{n}_eps_{epsilon}.pdf",
        std_scale=std_scale,
    )

    # 2) E[max Q]
    plot_with_std(
        t_grid=t_grid,
        mean1=mean_max_crp,
        std1=std_max_crp,
        mean2=mean_max_noncrp,
        std2=std_max_noncrp,
        ylabel=r"$\mathbb{E}\!\left[\max_{0\leq t\leq T}\|Q_t\|_1 \right]$", 
        title=fr"$\epsilon={epsilon}$",
        label1=fr"CRP",
        label2=fr"Non-CRP",
        pdf_name=f"mean_running_max_n_{n}_eps_{epsilon}.pdf",
        std_scale=std_scale,
    )

    # 3) E[max Q]/log(t+1)
    plot_with_std(
        t_grid=t_grid,
        mean1=mean_max_crp / log_t,
        std1=std_max_crp / log_t,
        mean2=mean_max_noncrp / log_t,
        std2=std_max_noncrp / log_t,
        ylabel=r"$\mathbb{E}[\max_{s\leq t} Q_s]/\log(t+1)$",
        title=fr"$\epsilon={epsilon}$",
        label1=fr"single face",
        label2=fr"non single face",
        pdf_name=f"mean_running_max_over_log_n_{n}_eps_{epsilon}.pdf",
        std_scale=std_scale,
    )


# example
run_comparison(
    n=4,
    T=100000,
    num_runs=200,
    epsilon=0.01,
    std_scale=0.25,
)